# DS6 — Feature Extraction

**Dataset:** Dataset 6 — Danish adolescents  
**Stage:** 1 — Dataset-specific feature extraction (no cross-dataset harmonization)  
**Input files:** `outputs/cleaned/ds6_*.csv`  
**Output files:** `outputs/features/ds6/`

---

## ⚠️ Critical structural note

DS6 has **ONLY `openedApp` events** in the resolution column.  
No dismiss or close outcomes are directly observed.  
**Do NOT treat DS6 outcome rates as equivalent to DS1/DS4/DS5 outcome rates.**

Behavioral response to friction in DS6 is instead captured via:
- Volume and session rate changes across study_period (baseline vs intervention)
- Sequence transitions (app_next, self-transitions)
- Longitudinal event rate slopes

---

## Outputs produced

| File | Rows | Contents |
|---|---|---|
| `ds6_behavior_features.csv` | ~243 | All behavioral participants + demographics |
| `ds6_survey_features.csv` | ~209+ | Wave 1 survey participants with behavioral data |
| `ds6_prepost_features.csv` | ~209 | Participants with behavioral + both survey waves |

---

## Event definitions

- **What counts as an event row:** Every row = one app-opening event (openedApp).
- **Intervention exposure:** `intervention_duration_sec > 0` (friction shown)
- **Outcome-bearing rows:** All rows (but all outcome = 'opened'; not informative for outcome rates)
- **Period:** `study_period` ∈ {baseline, intervention} — explicit in each row
- **Longitudinal axis:** `participation_day` and `study_week`

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("__file__").resolve().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

CLEANED = PROJECT_ROOT / "outputs" / "cleaned"
OUT = PROJECT_ROOT / "outputs" / "features" / "ds6"
OUT.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Output folder:", OUT)

## 1. Load cleaned files

In [ ]:
from src.features.ds6_features import load_ds6_files

beh, survey, demo, linkage = load_ds6_files(CLEANED)

print(f"Behavioral rows:         {len(beh):,}")
print(f"Behavioral participants:  {beh['participant_id_canonical'].nunique():,}")
print(f"Survey rows:             {len(survey):,}")
print(f"Demographic rows:        {len(demo):,}")
print(f"Linkage rows:            {len(linkage):,}")

print("\nStudy period distribution:")
print(beh['study_period'].value_counts())

print("\nStudy condition distribution:")
print(beh['study_condition'].value_counts())

In [ ]:
# Confirm DS6's single-outcome structure
print("Resolution values in DS6 (should be ONLY openedApp):")
print(beh['event_outcome_simple'].value_counts(dropna=False))
print()
print("⚠️  DS6 has no dismiss/close outcomes — see structural note in header.")

## 2. Build behavior features

Feature groups:
- **G1** Volume by study period (DIRECT, High — DS6-unique: baseline vs intervention)
- **G2** Inter-event timing (DIRECT, High — mean_time_diff_secs only; the
  group_id-based "session structure" features were removed in Stage 1 QC
  2026-08-05 — group_id does not group events into real sessions, see
  `ds6_timing_features()` docstring in `src/features/ds6_features.py`)
- **G3** Intervention exposure + type profile (DIRECT, High)
- **G4** Sequence / substitution via app_next/prev (NEAR-DIRECT, High — DS6-unique)
- **G5** App structure overall (DIRECT, High)
- **G6** App structure by period (DIRECT, High)
- **G7** Temporal structure (DIRECT-DERIVED, High)
- **G8** Longitudinal adaptation via participation_day (PROXY, Medium)
- Demographics: age, gender, region (DIRECT)

⚠️ No outcome rate features are computed — all outcomes = 'opened' in DS6.

In [ ]:
from src.features.ds6_features import build_ds6_behavior_features

print("Building behavioral features...")
beh_feats = build_ds6_behavior_features(beh, demo)
print(f"Shape: {beh_feats.shape}")
beh_feats.head(3)

In [ ]:
# Validate
assert beh_feats['participant_id_canonical'].nunique() == len(beh_feats)
print(f"Participants: {len(beh_feats):,}")

print("\nPeriod volume ratio (intervention / baseline events per day):")
print(beh_feats['event_rate_ratio_intervention_vs_baseline'].describe().round(3))

print("\nInter-event timing (mean_time_diff_secs; group_id-based session columns")
print("removed in Stage 1 QC 2026-08-05 -- group_id does not group real sessions):")
print(beh_feats['mean_time_diff_secs'].describe().round(3))

In [ ]:
print("Sequence features — substitution proxies:")
print(beh_feats[['self_transition_rate', 'cross_app_transition_rate',
                  'app_transition_entropy', 'max_consecutive_same_app']].describe().round(3))

In [ ]:
# F194: event_rate_ratio_intervention_vs_baseline is outlier-dominated -- the control
# mean of 2.000 printed here previously came entirely from participant 2HpKvJmJCr0lma4
# (n_baseline_days = 1, ratio 77.08), which reads as "the control arm doubled its usage".
# modeling_cautions.md sec.2 requires n_baseline_days >= 7 before this feature is used, so
# report the raw and filtered pictures plus the median alongside the mean.
cols = ['events_per_day_baseline', 'events_per_day_intervention',
        'event_rate_ratio_intervention_vs_baseline', 'self_transition_rate']
print("Condition breakdown, UNFILTERED means (ratio is outlier-dominated -- do not quote):")
print(beh_feats.groupby('study_condition')[cols].mean().round(3))
print("\nCondition breakdown, UNFILTERED medians:")
print(beh_feats.groupby('study_condition')[cols].median().round(3))
_f = beh_feats[beh_feats['n_baseline_days'] >= 7]
print(f"\nCondition breakdown after the required n_baseline_days >= 7 filter "
      f"(n={len(_f)} of {len(beh_feats)}):")
print(_f.groupby('study_condition')[cols].mean().round(3))

In [ ]:
out_path = OUT / "ds6_behavior_features.csv"
beh_feats.to_csv(out_path, index=False)
print(f"Saved: {out_path}  ({len(beh_feats):,} rows, {beh_feats.shape[1]} columns)")

## 3. Build survey features (two waves)

Feature groups:
- **G10** Wave 1 psychometric indices: addiction, self-control, FoMO, wellbeing (DIRECT, High)
- **G11** Wave 2 indices where available (NaN for participants missing wave 2) (DIRECT, High)

In [ ]:
from src.features.ds6_features import build_ds6_survey_features

survey_feats = build_ds6_survey_features(beh, survey)
print(f"Shape: {survey_feats.shape}")
print(f"Participants with wave 2: {survey_feats['has_wave2_survey'].sum():,}")

print("\nPre-survey indices by condition:")
pre_cols = ['addiction_index_pre', 'self_control_index_pre', 'fo_mo_pre', 'trivsel_pre']
print(survey_feats.groupby('study_condition')[pre_cols].mean().round(3))

In [ ]:
out_path = OUT / "ds6_survey_features.csv"
survey_feats.to_csv(out_path, index=False)
print(f"Saved: {out_path}  ({len(survey_feats):,} rows, {survey_feats.shape[1]} columns)")

## 4. Build pre-post survey change features

Feature groups:
- **G12** Pre-to-post change scores for all indices (DIRECT-DERIVED, Medium)  
Available for: ~209 participants with behavioral + both survey waves

In [ ]:
from src.features.ds6_features import build_ds6_prepost_features

prepost_feats = build_ds6_prepost_features(beh, survey)
print(f"Shape: {prepost_feats.shape}")

change_cols = [c for c in prepost_feats.columns if '_change' in c]
print("\nChange score distributions by condition:")
print(prepost_feats.groupby('study_condition')[change_cols].mean().round(3))

In [ ]:
out_path = OUT / "ds6_prepost_features.csv"
prepost_feats.to_csv(out_path, index=False)
print(f"Saved: {out_path}  ({len(prepost_feats):,} rows, {prepost_feats.shape[1]} columns)")

## 5. Output summary

In [ ]:
print("=" * 60)
print("DS6 Feature Extraction — Output Summary")
print("=" * 60)

for fname, df in [
    ('ds6_behavior_features.csv', beh_feats),
    ('ds6_survey_features.csv', survey_feats),
    ('ds6_prepost_features.csv', prepost_feats),
]:
    n_features = df.shape[1] - 1
    print(f"  {fname}")
    print(f"    {len(df):,} participants × {n_features} features")
    missing_any = df.drop(columns='participant_id_canonical').isna().any(axis=1).sum()
    print(f"    Participants with any NaN feature: {missing_any:,}")
    print()

print()
print("REMINDER: DS6 behavioral features do NOT include dismiss/close outcome rates.")
print("All DS6 events are openedApp. Use volume/session/sequence changes as behavioral proxies.")